# 00 API checks: SDV 2.0.0 / SDMetrics 0.32.0 compatibility evidence


| Section | Question answered |
|---|---|
| A | How is metadata built in SDV 2.0? |
| B | Which synthesizer constructor arguments changed? |
| C | What data format do `fit` and `sample` expect? |
| D | How are synthesizers saved and reloaded? |
| E | How is conditional (per-class) sampling called? |
| F | Where do the quality-report functions live? |
| G | Why does Column Pair Trends return NaN? |


In [1]:
import inspect
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import sdmetrics
import sdv
import torch

FEATURES = ["nevents", "ndays_act", "ncontent", "nforum_posts"]
TARGET = "Engagement_Level"
TABLE = "students"
SEED = 42


def find_project_root(start=None):
    """Walk upwards until a folder containing configs/base.yaml is found."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "configs" / "base.yaml").exists():
            return folder
    raise FileNotFoundError(f"Could not find configs/base.yaml above {start}")


def unwrap(result):
    """Return a DataFrame whether SDV gave us a DataFrame or {table: DataFrame}."""
    if isinstance(result, dict):
        return result[TABLE] if TABLE in result else next(iter(result.values()))
    return result


PROJECT_ROOT = find_project_root()
gen_dir = PROJECT_ROOT / "artifacts" / "generators"
real_train = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "train.csv")[FEATURES]

print("sdv:", sdv.__version__, "| sdmetrics:", sdmetrics.__version__, "| torch:", torch.__version__)
print("Project root:", PROJECT_ROOT)
print("real_train:", real_train.shape)

sdv: 2.0.0 | sdmetrics: 0.32.0 | torch: 2.14.1+cpu
Project root: D:\student_engagement_twin
real_train: (21328, 4)


## A. Metadata API

**Question:** SDV 1.x built metadata with `SingleTableMetadata().detect_from_dataframe(df)`. What does 2.0.0 offer?

**Method:** list the public names in `sdv.metadata`, check which `detect_*` methods `Metadata` has, build metadata from real train features, and print the column types and the `update_column` signature.

**Expected:** `SingleTableMetadata` absent, `detect_from_dataframes` present, all four features `numerical`.

In [2]:
from sdv import metadata as md
from sdv.metadata import Metadata

print("Public names in sdv.metadata:", [n for n in dir(md) if not n.startswith("_")])
print("SingleTableMetadata available      :", hasattr(md, "SingleTableMetadata"))
print("Metadata.detect_from_dataframe (1.x):", hasattr(Metadata, "detect_from_dataframe"))
print("Metadata.detect_from_dataframes    :", hasattr(Metadata, "detect_from_dataframes"))

metadata = Metadata.detect_from_dataframes({TABLE: real_train})
cols = metadata.to_dict()["tables"][TABLE]["columns"]
print("Detected sdtypes:", {c: v["sdtype"] for c, v in cols.items()})
print("update_column signature:", inspect.signature(Metadata.update_column))

Public names in sdv.metadata: ['InvalidMetadataError', 'Metadata', 'MetadataNotFittedError', 'errors', 'metadata', 'metadata_upgrader', 'utils', 'visualization']
SingleTableMetadata available      : False
Metadata.detect_from_dataframe (1.x): False
Metadata.detect_from_dataframes    : True
Detected sdtypes: {'nevents': 'numerical', 'ndays_act': 'numerical', 'ncontent': 'numerical', 'nforum_posts': 'numerical'}
update_column signature: (self, column_name, table_name=None, **kwargs)


## B. Synthesizer constructor arguments

**Question:** the SDV 1.x argument `cuda=` raised a `TypeError`. Which arguments do `CTGANSynthesizer` and `TVAESynthesizer` accept in 2.0.0, and what is the default of CTGAN's `log_frequency`?

**Why it matters:** `log_frequency=True` (the default) distorted the synthetic class share (0.54 / 0.46 instead of 0.82 / 0.18), so the default is part of the evidence.

**Expected:** `cuda` rejected, `enable_gpu` accepted, `log_frequency` default `True`.

In [3]:
from sdv.single_table import CTGANSynthesizer, TVAESynthesizer

for cls in (CTGANSynthesizer, TVAESynthesizer):
    params = inspect.signature(cls.__init__).parameters
    print(f"{cls.__name__}: {list(params)}")
    print("   'cuda' accepted      :", "cuda" in params)
    print("   'enable_gpu' accepted:", "enable_gpu" in params)

lf = inspect.signature(CTGANSynthesizer.__init__).parameters.get("log_frequency")
print("\nCTGAN log_frequency default:", lf.default if lf else "parameter not found")

CTGANSynthesizer: ['self', 'metadata', 'enforce_min_max_values', 'enforce_rounding', 'locales', 'embedding_dim', 'generator_dim', 'discriminator_dim', 'generator_lr', 'generator_decay', 'discriminator_lr', 'discriminator_decay', 'batch_size', 'discriminator_steps', 'log_frequency', 'verbose', 'epochs', 'pac', 'enable_gpu']
   'cuda' accepted      : False
   'enable_gpu' accepted: True
TVAESynthesizer: ['self', 'metadata', 'enforce_min_max_values', 'enforce_rounding', 'embedding_dim', 'compress_dims', 'decompress_dims', 'l2scale', 'batch_size', 'verbose', 'epochs', 'loss_factor', 'enable_gpu']
   'cuda' accepted      : False
   'enable_gpu' accepted: True

CTGAN log_frequency default: True


## C. Data format for `fit` and `sample`

**Question:** `synth.fit(df)` raised `InvalidDataTypeError` ("must be a dictionary containing exactly one table name mapped to a pandas DataFrame"). What does `fit` accept, and what does `sample` return?

**Method:** train a throwaway CTGAN for 1 epoch on 500 complete real-train rows (in memory only), call `fit` with a DataFrame and then with `{table: DataFrame}`, and report the type returned by `sample`.

**Expected:** DataFrame rejected, dictionary accepted. The type returned by `sample` is recorded from the output, not assumed.

In [4]:
np.random.seed(SEED)
torch.manual_seed(SEED)

small = real_train.dropna().sample(500, random_state=SEED).reset_index(drop=True)
small_meta = Metadata.detect_from_dataframes({TABLE: small})

attempt = CTGANSynthesizer(small_meta, epochs=1, verbose=False)
try:
    attempt.fit(small)
    print("fit(DataFrame)       : accepted")
except Exception as e:
    print("fit(DataFrame)       : rejected ->", type(e).__name__)

tiny = CTGANSynthesizer(small_meta, epochs=1, verbose=False)   # fresh object
tiny.fit({TABLE: small})
print("fit({table: df})     : accepted")

out = tiny.sample(num_rows=10)
print("sample() returns     :", type(out).__name__)
print("shape after unwrap   :", unwrap(out).shape)

d:\student_engagement_twin\.venv\Lib\site-packages\sdv\single_table\base.py:136: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


fit(DataFrame)       : rejected -> InvalidDataTypeError
fit({table: df})     : accepted
sample() returns     : dict
shape after unwrap   : (10, 4)


## D. Saving and reloading synthesizers

**Question:** `CTGANSynthesizer.load(path)` raised `AttributeError` in 2.0.0. What replaces it, and does it work for **both** saved generators? (Earlier, only the CTGAN file had been verified.)

**Method:**
1. Check whether `.load` and `.save` exist on the class.
2. Round-trip the throwaway model from Section C through a temporary folder with `sdv.utils.load_synthesizer`. **This section depends on Section C.**
3. Reload the real saved `ctgan_random_target.pkl` and `tvae_random_target.pkl` (read-only).

**Expected:** `.load` absent, `.save` present, all three reloads succeed.

In [5]:
import sdv.utils as sutils
from sdv.utils import load_synthesizer

print("sdv.utils load/save names:",
      [n for n in dir(sutils) if "load" in n.lower() or "save" in n.lower()])
print("CTGANSynthesizer.load exists:", hasattr(CTGANSynthesizer, "load"))
print("CTGANSynthesizer.save exists:", hasattr(CTGANSynthesizer, "save"))

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "tiny.pkl"
    tiny.save(str(path))
    print("round trip (throwaway):", type(load_synthesizer(str(path))).__name__)

for name in ("ctgan", "tvae"):
    path = gen_dir / f"{name}_random_target.pkl"
    if path.exists():
        print(f"reload {path.name}:", type(load_synthesizer(str(path))).__name__)
    else:
        print(f"reload {path.name}: file not found, skipped")

sdv.utils load/save names: ['__loader__', 'load_constraints', 'load_synthesizer']
CTGANSynthesizer.load exists: False
CTGANSynthesizer.save exists: True
round trip (throwaway): CTGANSynthesizer
reload ctgan_random_target.pkl: CTGANSynthesizer
reload tvae_random_target.pkl: TVAESynthesizer


## E. Conditional (per-class) sampling

**Question:** the paper balances classes by conditional sampling. How is a condition built in 2.0.0, and does `sample_from_conditions` return the requested class?

**Method:** print the signatures of `Condition`, `DataFrameCondition` and `sample_from_conditions`. Then reload each saved target-variant generator (trained on real train only) and request 50 rows of class 0 and 50 of class 1.

**Expected:** every returned row has the requested class. Anything else is a library limitation to document, not to work around silently.

In [6]:
from sdv.sampling import Condition, DataFrameCondition

print("Condition             :", inspect.signature(Condition.__init__))
print("DataFrameCondition    :", inspect.signature(DataFrameCondition.__init__))
print("sample_from_conditions:", inspect.signature(CTGANSynthesizer.sample_from_conditions))

for name in ("ctgan", "tvae"):
    path = gen_dir / f"{name}_random_target.pkl"
    if not path.exists():
        print(f"{name}: file not found, skipped")
        continue
    synth = load_synthesizer(str(path))
    for label in (0, 1):
        cond = Condition(column_values={TARGET: label}, num_rows=50)
        out = unwrap(synth.sample_from_conditions(conditions=[cond]))
        print(f"{name} | requested {label}: shape={out.shape}, "
              f"returned={out[TARGET].value_counts().to_dict()}")

Condition             : (self, column_values, num_rows=1, table_name=None)
DataFrameCondition    : (self, dataframe, table_name=None)
sample_from_conditions: (self, conditions, max_tries_per_batch=100, batch_size=None, output_folder_path=None)


Sampling conditions: 100%|██████████| 50/50 [00:00<00:00, 204.90it/s]


ctgan | requested 0: shape=(50, 5), returned={0: 50}


Sampling conditions: 100%|██████████| 50/50 [00:00<00:00, 247.05it/s]


ctgan | requested 1: shape=(50, 5), returned={1: 50}


Sampling conditions: 100%|██████████| 50/50 [00:00<00:00, 469.73it/s]


tvae | requested 0: shape=(50, 5), returned={0: 50}


Sampling conditions: 100%|██████████| 50/50 [00:00<00:00, 261.47it/s]

tvae | requested 1: shape=(50, 5), returned={1: 50}


## F. Evaluation API location

**Question:** `from sdv.evaluation.single_table import evaluate_quality` raised `ImportError` in 2.0.0. Where does it live, and are the SDMetrics classes used in this project importable?

**Expected:** `evaluate_quality` importable from `sdv.evaluation`; `MissingValueSimilarity` and `CorrelationSimilarity` importable from `sdmetrics`.

In [7]:
import sdv.evaluation as sev

print("Public names in sdv.evaluation:", [n for n in dir(sev) if not n.startswith("_")])

from sdv.evaluation import evaluate_quality
print("evaluate_quality signature:", inspect.signature(evaluate_quality))

from sdmetrics.column_pairs import CorrelationSimilarity
from sdmetrics.single_column import MissingValueSimilarity
print("CorrelationSimilarity and MissingValueSimilarity: importable")

Public names in sdv.evaluation: ['evaluate_quality', 'evaluation', 'get_cardinality_plot', 'get_column_pair_plot', 'get_column_plot', 'run_diagnostic']
evaluate_quality signature: (real_data, synthetic_data, metadata, verbose=True)
CorrelationSimilarity and MissingValueSimilarity: importable


## G. Why is Column Pair Trends NaN?

**Question:** `evaluate_quality` returned `NaN` for Column Pair Trends, so its overall score equalled Column Shapes. Is the data or the library at fault?

**Method (nothing saved):** compare real train with the restored CTGAN baseline output in three ways:
1. SDV's quality report, with per-pair details.
2. `CorrelationSimilarity` called directly on one pair.
3. Plain pandas correlation for the same pair.

**Reading the result:**
- Real correlation computed but synthetic correlation `NaN` in the report, while the direct call returns a score: the fault is in the report in this version combination.
- Both pandas values are numbers: the data itself is fine.

**Consequence:** pair trends are computed manually in `05_fidelity.ipynb`.

In [8]:
fake_path = gen_dir / "ctgan_random_restored.csv"

if not fake_path.exists():
    print("ctgan_random_restored.csv not found, section skipped")
else:
    fake = pd.read_csv(fake_path)[FEATURES]
    meta = Metadata.detect_from_dataframes({TABLE: real_train})

    report = evaluate_quality({TABLE: real_train}, {TABLE: fake}, meta, verbose=False)
    print(report.get_properties().to_string(index=False))
    details = report.get_details("Column Pair Trends")
    print(details[["Column 1", "Column 2", "Score",
                   "Real Correlation", "Synthetic Correlation"]].to_string(index=False))

    pair = ["nevents", "ndays_act"]
    direct = CorrelationSimilarity.compute(real_train[pair].dropna(), fake[pair].dropna())
    print("\nDirect CorrelationSimilarity (nevents, ndays_act):", round(float(direct), 4))
    print("pandas corr  real:", round(real_train[pair].corr().iloc[0, 1], 3),
          "| fake:", round(fake[pair].corr().iloc[0, 1], 3))

          Property    Score
     Column Shapes 0.894222
Column Pair Trends      NaN
 Column 1     Column 2  Score  Real Correlation  Synthetic Correlation
  nevents    ndays_act    NaN          0.264074                    NaN
  nevents     ncontent    NaN          0.197468                    NaN
  nevents nforum_posts    NaN          0.175642                    NaN
ndays_act     ncontent    NaN          0.272695                    NaN
ndays_act nforum_posts    NaN          0.310513                    NaN
 ncontent nforum_posts    NaN          0.140820                    NaN

Direct CorrelationSimilarity (nevents, ndays_act): 0.9611
pandas corr  real: 0.264 | fake: 0.342


## Summary: library incompatibilities (for the README)

Environment: Python 3.11.9, SDV 2.0.0, SDMetrics 0.32.0. Verified by the sections above.

| # | Topic | SDV 1.x behaviour | Installed behaviour | Workaround in this project |
|---|---|---|---|---|
| 1 | Metadata | `SingleTableMetadata().detect_from_dataframe(df)` | Class removed. `Metadata.detect_from_dataframes({table: df})` | Section A call |
| 2 | GPU argument | `cuda=` accepted | `TypeError`. Use `enable_gpu` | Argument omitted (CPU/default) |
| 3 | Data format | `fit(df)` | `fit` requires `{table: df}` | Dictionary form, with `unwrap()` on outputs |
| 4 | Reloading | `Synthesizer.load(path)` | Class method removed. Use `sdv.utils.load_synthesizer` | Section D call |
| 5 | Evaluation import | `sdv.evaluation.single_table` | Module removed. Use `sdv.evaluation.evaluate_quality` | Section F import |
| 6 | Column Pair Trends | Numeric score | `NaN` in the report. `CorrelationSimilarity` works directly | Mean over the six feature pairs, computed manually |
| 7 | CTGAN class balance | n/a | Default `log_frequency=True` flattened the class share (0.54 / 0.46 vs real 0.82 / 0.18) | `log_frequency=False` for unbalanced runs |

**Verified in this run:** `sample()` returns a dictionary (`{table: DataFrame}`), so `unwrap()` is required, not optional. `sdv.utils.load_synthesizer` reloads both saved target-variant generators (`ctgan_random_target.pkl` and `tvae_random_target.pkl`). Conditional sampling on TVAE returns exactly the requested class (50 of 50 per class). CTGAN conditional sampling was verified earlier (200 of 200 per class).

**Hardware:** CPU only (`torch 2.14.1+cpu`). Training times on this machine: CTGAN 6.0 to 7.5 min, TVAE 2.0 to 3.0 min for 300 epochs on 21,328 rows.

**Limitation:** manual pair trends follow the same idea as SDV's report but are not guaranteed identical to it. The cause of the report's `NaN` was isolated to the report itself (direct `CorrelationSimilarity` returns a score, pandas correlations on the synthetic data are numeric), but the underlying library bug was not investigated further.
**Limitation:** manual pair trends follow the same idea as SDV's report but are not guaranteed identical to it.